# Comparing the existing MORL route models

This notebook explains the paired comparison saved with TI-AI-11. It reads the saved results; it does **not** train or rerun either model. Run the cells from top to bottom with Python, pandas and matplotlib installed. The notebook can be launched from this folder or any folder inside the repository.

## What was compared

- **Scalarized (PIC A):** one preference-conditioned model; its selected checkpoint is the warm start (update 0).
- **Fixed set (PIC B):** five policies; each request uses the nearest preference member.
- **Regret prior:** an untrained baseline, included to show the contribution of the direct regret path.
- The fastest route is the common reference; the exact weighted optimum is an additional benchmark.

Each method sees the same graph, origin–destination pair, time window, exposure field and requested preference. The primary group is 200 bike trips of 0.5–3 km on real IDW fields. Another 50 pairs cover longer trips (3–8 km). Each group also has a separately reported synthetic perturbation. The 11 requested preferences run from 0 to 1; α=0.5 is the default.

**Interpretation:** TI-AI-10's test scenarios had already been inspected; this is an exploratory comparison, not a fresh blind test. The two candidates differ in the regret path as well as preference handling, so this comparison does not isolate the effect of preference conditioning. The synthetic field results are not measurements of real-world exposure savings.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo(start):
    for candidate in (start, *start.parents):
        if (candidate / 'docs' / 'experiments' / 'TI-AI-11' / 'morl-comparison' / 'summary.json').is_file():
            return candidate
    raise FileNotFoundError('Open the notebook from inside the SIPPolusi repository.')

root = find_repo(Path.cwd().resolve())
evidence = root / 'docs' / 'experiments' / 'TI-AI-11' / 'morl-comparison'
summary = json.loads((evidence / 'summary.json').read_text(encoding='utf-8'))
decisions = json.loads((evidence / 'decision.json').read_text(encoding='utf-8'))
manifest = json.loads((evidence / 'manifest.json').read_text(encoding='utf-8'))
names = {'scalarized': 'Scalarized (PIC A)', 'fixed_preference_set': 'Fixed set (PIC B)',
         'regret_prior': 'Regret prior'}
order = list(names)
print('Evaluation commit:', manifest['code']['commit'])
print('Graph:', manifest['graph_version'], '| mode:', manifest['mode'], '| split:', manifest['split'])
print('Selected scalarized update:', manifest['candidates']['scalarized']['selected_update'])


## Primary results: real fields, 0.5–3 km

Success and latency use **decoded routes**, as in the original protocol: a fixed-set member is decoded once for all preferences mapped to that member, while the scalarized model is decoded once per requested preference. All returned routes were checked for validity. Timings are local CPU measurements, not Railway service latency, and are not hard timeouts.

Exposure and time changes below are medians against the fastest route at α=0.5; failed policy routes are excluded from those particular changes. The summary also retains results with failures counted as fastest-route fallbacks. The EAI-04 target is at least 4% median exposure reduction with no more than 20% median added travel time.


In [ ]:
primary = summary['trained-range']['real']
rows = []
for key in order:
    s = primary[key]
    default = s['by_alpha']['0.5']['policy']
    rows.append({'Method': names[key], 'Decoded routes': s['decoded_routes'],
                 'Success (%)': 100*s['success_rate'], 'Valid returned (%)': 100*s['route_validity'],
                 'p95 (ms)': s['inference_ms']['p95'], 'Dead ends': s['failures'].get('dead_end', 0),
                 'Median exposure change (%)': default['exposure']['median'],
                 'Median added time (%)': default['time']['median'],
                 'TI-AI-10 classification': decisions[key]['outcome']})
primary_table = pd.DataFrame(rows).set_index('Method')
print(primary_table.round(2).to_string())


All three methods pass the protocol's technical gates on the primary group, while none meets the 4% median exposure-reduction target. The exact weighted optimum also has 0% median reduction at α=0.5 on these fields. This indicates limited median opportunity in these scenarios; it is not proof that other fields offer none.

## Does the route respond to preference?

The EAI-05 fulfilment measure compares α=1 with α=0 on pairs that succeed at both endpoints. **Identical routes count as fulfilled**, so the additional “shifted in the intended direction” measure is essential. Pairs failing at an endpoint are omitted from both rates and are counted in the table's usable denominator.


In [ ]:
pref = []
for key in order:
    p = primary[key]['preference']
    pref.append({'Method': names[key], 'Usable pairs': p['usable'],
                 'Fulfilled (%)': 100*p['fulfilled_rate'], 'Shifted in intended direction (%)': 100*p['shifted_rate']})
preference_table = pd.DataFrame(pref).set_index('Method')
print(preference_table.round(2).to_string())
ax = preference_table[['Fulfilled (%)', 'Shifted in intended direction (%)']].plot.bar(
    figsize=(9, 4), color=['#5686a5', '#e49a49'], rot=0)
ax.axhline(80, color='#a13c3c', linestyle='--', label='EAI-05 target (fulfilment)')
ax.set_ylabel('Share of usable pairs (%)')
ax.set_ylim(0, 105)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Robustness: longer trips and synthetic fields

The longer-trip group is outside the models' training distance range and is descriptive. Synthetic fields test the ability to react when a stronger trade-off is present; they do not establish real exposure improvements. Compare both success and preference response: a route that fails is not a successful model route, even when a service would fall back to the fastest route.


In [ ]:
robustness = []
for stratum, field in [('trained-range', 'real'), ('trained-range', 'perturbed'),
                       ('longer-trips', 'real'), ('longer-trips', 'perturbed')]:
    for key in order:
        s = summary[stratum][field][key]
        p = s['preference']
        robustness.append({'Trips': stratum, 'Field': field, 'Method': names[key],
                           'Decoded routes': s['decoded_routes'], 'Success (%)': 100*s['success_rate'],
                           'p95 (ms)': s['inference_ms']['p95'], 'Dead ends': s['failures'].get('dead_end', 0),
                           'Usable endpoint pairs': p['usable'],
                           'Shifted (%)': 100*p['shifted_rate'] if p['shifted_rate'] is not None else None})
robustness_table = pd.DataFrame(robustness)
print(robustness_table.round(2).to_string(index=False))


## What this means for selection

On this run, both candidates are **conditional candidates** under TI-AI-10: they meet the primary technical gates but do not meet all research targets. PIC A passes the recorded preference-fulfilment criterion and has better longer-trip success, but few primary real-field pairs actually shift in the intended direction. PIC B already has packaging and backend support; selecting PIC A would require its own package and backend loader support.

The regret prior is a baseline, not a third trained candidate. A definitive selection also needs an agreed decision rule and a fresh held-out comparison, preferably on fields from the registered spatial models. The manifest contains the checkpoint, graph and field identities and the recorded criteria. The gzip file contains paired per-scenario measurements and route hashes; it is not a full route geometry archive.

### Evidence files

- `manifest.json`: run configuration, commit and artifact identities.
- `summary.json` and `decision.json`: per-group metrics and protocol classifications.
- `scenario_od_pairs.json`: sampled origin–destination pairs.
- `scenarios.jsonl.gz`: paired per-preference measurements, failures and route hashes.
